# 🚀 Huấn Luyện Connect 4 RL Trên Google Colab

Notebook này được thiết kế để **giải phóng 100% tài nguyên cho laptop của bạn**:
- Toàn bộ quá trình huấn luyện Curriculum PPO (16 agents, hàng trăm ngàn bước) chạy trên **GPU/CPU đám mây miễn phí của Google**.
- Checkpoint được **tự động lưu vào Google Drive** (không lo mất mô hình nếu rớt mạng).
- Sau khi train xong, **tải gói zip về laptop với 1 click** để chơi trên giao diện Pygame đồ họa mượt mà!

--- 
## 1️⃣ Bước 1: Cài đặt thư viện

In [ ]:
!pip install -q gymnasium stable-baselines3 pygame tqdm tensorboard torch

--- 
## 2️⃣ Bước 2: Kết nối Google Drive & Mã nguồn dự án

In [ ]:
from google.colab import drive
import os

# Mount Drive để lưu checkpoints an toàn
drive.mount('/content/drive')

# Clone mã nguồn dự án từ GitHub (nếu chưa có)
%cd /content
if not os.path.exists('DO_AN_TRI_TUE_NHAN_TAO_NANG_CAO_SGU_2026'):
    !git clone https://github.com/ducnhan123-co/DO_AN_TRI_TUE_NHAN_TAO_NANG_CAO_SGU_2026.git

%cd /content/DO_AN_TRI_TUE_NHAN_TAO_NANG_CAO_SGU_2026/drmkn_connect4_source/drmkn_connect4_src

# Chế độ headless cho máy ảo Colab
os.environ["SDL_VIDEODRIVER"] = "dummy"
print("✓ Môi trường huấn luyện đã sẵn sàng!")

--- 
## 3️⃣ Bước 3: Huấn Luyện Chuỗi Curriculum Agents (8x9 hoặc 6x7)
Mỗi agent sau khi hoàn thành sẽ được lưu **ngay lập tức** vào Google Drive.
Nếu bạn bị gián đoạn và chạy lại, code sẽ tự động nhận diện và **tiếp tục từ agent kế tiếp** mà không train lại từ đầu!

In [ ]:
import os
from connect4 import Connect4
from stable_baselines3 import PPO

def train_curriculum_colab(
    rows=8,
    cols=9,
    total_timesteps_per_agent=100_000,
    num_agents=16,
    save_drive_dir='/content/drive/MyDrive/Connect4_Models/saved_agents_8x9'
):
    os.makedirs(save_drive_dir, exist_ok=True)
    tb_log_dir = "./logs/curriculum_colab"
    os.makedirs(tb_log_dir, exist_ok=True)

    env = Connect4(rows=rows, cols=cols)
    env.reset()

    print(f"Thư mục lưu trữ: {save_drive_dir}")
    print(f"Bắt đầu Curriculum Learning cho bàn cờ {rows}x{cols} ({num_agents} agents)...\n")

    # --- Agent 0: Đối đầu Random ---
    agent0_path = os.path.join(save_drive_dir, "agent0.zip")
    if os.path.exists(agent0_path):
        print("[✓] Đã tồn tại agent0 trên Google Drive, bỏ qua...")
    else:
        print(f"=== [{rows}x{cols}] Huấn luyện agent0 (đối đầu random) ===")
        agent0 = PPO("MlpPolicy", env, verbose=1, tensorboard_log=tb_log_dir)
        agent0.learn(total_timesteps=total_timesteps_per_agent, progress_bar=True, tb_log_name=f"PPO_agent0_{rows}x{cols}")
        agent0.save(os.path.join(save_drive_dir, "agent0"))
        print("===> Hoàn tất và đã lưu agent0 vào Google Drive!\n")

    # --- Agent 1 đến num_agents-1: Đối đầu Agent trước đó ---
    for i in range(1, num_agents):
        current_agent_path = os.path.join(save_drive_dir, f"agent{i}.zip")
        if os.path.exists(current_agent_path):
            print(f"[✓] Đã tồn tại agent{i} trên Google Drive, bỏ qua...")
            continue

        print(f"=== [{rows}x{cols}] Huấn luyện agent{i} (đối đầu agent{i-1}) ===")
        prev_agent_path = os.path.join(save_drive_dir, f"agent{i-1}")
        agent_prev = PPO.load(prev_agent_path)

        agent_i = PPO("MlpPolicy", env, verbose=1, tensorboard_log=tb_log_dir)
        vec_env = agent_i.get_env()
        vec_env.env_method("set_opp_agent", agent_prev, 0.2)
        agent_i.learn(
            total_timesteps=total_timesteps_per_agent,
            progress_bar=True,
            tb_log_name=f"PPO_agent{i}_{rows}x{cols}",
        )
        agent_i.save(os.path.join(save_drive_dir, f"agent{i}"))
        print(f"===> Hoàn tất và đã lưu agent{i} vào Google Drive!\n")

    print("\n🎉 TẤT CẢ AGENT ĐÃ HOÀN TẤT HUẤN LUYỆN!")

# Khởi động quá trình train (mặc định bàn 8x9, 16 agents):
train_curriculum_colab(
    rows=8,
    cols=9,
    total_timesteps_per_agent=100_000,
    num_agents=16,
    save_drive_dir='/content/drive/MyDrive/Connect4_Models/saved_agents_8x9'
)

--- 
## 4️⃣ Bước 4: Nén và Tải về Laptop
Chạy ô này để tải file `saved_agents_8x9.zip` về máy tính của bạn.

Sau khi tải về, bạn chỉ việc:
1. Giải nén vào thư mục `drmkn_connect4_source/drmkn_connect4_src/saved_agents_8x9/` trên laptop.
2. Chạy `python play.py` trên laptop để mở game Pygame đồ họa mượt mà!

In [ ]:
from google.colab import files
import shutil

src_dir = '/content/drive/MyDrive/Connect4_Models/saved_agents_8x9'
zip_name = '/content/saved_agents_8x9'

print("Đang nén file mô hình...")
shutil.make_archive(zip_name, 'zip', src_dir)
print("✓ Nén thành công! Bắt đầu tải xuống máy tính...")
files.download(f"{zip_name}.zip")